# FerroSim — Adaptive Piecewise-Linear Control (APLC)

**Standalone, reproducible, variable-topology optimization notebook**. This study introduces an evolved number of straight-line segments, independently positioned knot times, and variable knot amplitudes. It uses the same published 20×20 ferroic simulator implementation and 20 paired disorder maps as the Legendre/Fourier standalones. The physical reward is terminal spatial absolute polarization curl.

**Scientific honesty and comparison:** The GA and QD proposers use structural mutations. The sequential and denoising proposers are explicitly *inspired* by GFlowNets and diffusion; they do **not** implement a trained GFlowNet policy or score model. All four methods employ a trained neural-feature Gaussian-process (DKL) acquisition model, and all acquired rewards come from actual FerroSim simulator evaluations. This experiment therefore compares proposal strategies, not four exact reproductions of the previously trained generators.

**Execution:** Run top-to-bottom. Set `PILOT=True` for the fast 1-map, 1-seed, Euler integration smoke test; set `PILOT=False` for the 20-map, 3-seed, RK4 full experiment. No GitHub checkout or external files required. Outputs are saved next to this notebook in `APLC_experiment_runtime`.

## 1. Configuration and reproducible runtime

In [ ]:
import os,sys,json,pathlib,base64,zlib,hashlib,importlib,subprocess
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
PILOT=True  # CHANGE TO False FOR FULL RK4 20-DISORDER STUDY
RUN_EXPERIMENT=True
RUN_TRANSFER=False # expensive: reevaluate selected controls on every disorder
RUN_REFINEMENT=False # expensive: independent RK4 fidelity=2
RUN_SPATIAL_MAP=False # requires an additional RK4 evaluation
ROOT=Path.cwd()/'APLC_experiment_runtime';ROOT.mkdir(exist_ok=True)
for var in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','NUMBA_NUM_THREADS']:
    os.environ.setdefault(var,'1')
sys.path.insert(0,str(ROOT.resolve()))
print('Notebook runtime:', ROOT.resolve(), 'mode:', 'Euler pilot' if PILOT else 'RK4 full study')

## 2. Fully embedded numerical and optimization source code

All modules below are written by this notebook to its own runtime directory. Each block is readable and editable. The physics integrator is embedded, not downloaded at execution time.

In [ ]:
%%writefile APLC_experiment_runtime/representation.py
"""Variable-topology piecewise-linear control, physically zero slope and field at both ends."""
import numpy as np
from scipy.signal import find_peaks
DIM=32; MIN_K=4; MAX_K=16; DRIVE_SAMPLES=900

def decode(x):
    x=np.asarray(x,float).reshape(-1); K=int(np.clip(np.rint(x[0]),MIN_K,MAX_K))
    logits=np.r_[np.clip(x[1:K],-15,15),0.]
    durations=np.exp(logits-logits.max());durations/=durations.sum()
    times=np.r_[0.,np.cumsum(durations)];times[-1]=1.
    amps=np.zeros(K+1); amps[2:K-1]=x[18:18+K-3]  # amplitudes at interior knots 2..K-2
    return times,amps

def encode(times,amps):
    times=np.asarray(times,float);amps=np.asarray(amps,float);K=len(times)-1
    assert MIN_K<=K<=MAX_K and len(amps)==K+1
    d=np.diff(times);assert np.all(d>0)
    x=np.zeros(DIM);x[0]=K;x[1:K]=np.log(d[:-1]/d[-1]);x[18:18+K-3]=amps[2:K-1]
    return x

def canonical(x):
    t,a=decode(x);return encode(t,a)

def fields(x,n=900):
    x=np.atleast_2d(np.asarray(x,float));u=np.linspace(0,1,int(n))
    return np.array([np.interp(u,*decode(row)) for row in x])

def basis(n=96):
    # The manifold is nonlinear and variable dimensional; this method exists only
    # to prevent silent use of a linear-basis assumption by an external DKL.
    raise NotImplementedError('Variable-topology waves have no fixed linear basis')

def complexity(x,n=4097):
    z=np.atleast_2d(np.asarray(x,float));w=fields(z,n)
    out={k:np.zeros(len(z),int) for k in ['turns','raw_turns','turns_1pct','turns_5pct','segments','effective_segments']}
    for i,row in enumerate(z):
        t,a=decode(row);K=len(t)-1;out['segments'][i]=K
        # effective segments: number of distinct slope runs (within tight rel tolerance)
        slopes=np.diff(a)/np.diff(t)
        tol=1e-7*max(1.,np.max(np.abs(slopes)))
        out['effective_segments'][i]=1+int(np.sum(np.abs(np.diff(slopes))>tol))
        ptp=np.ptp(w[i]);
        if ptp>0:
            for key,frac in [('raw_turns',0),('turns_1pct',.01),('turns',.02),('turns_5pct',.05)]:
                out[key][i]=len(find_peaks(w[i],prominence=frac*ptp)[0])+len(find_peaks(-w[i],prominence=frac*ptp)[0])
    out['peak_ec']=np.max(np.abs(w),axis=1);out['rms_ec']=np.sqrt(np.mean(w*w,axis=1))
    return out

def sample_one(rng,k=None):
    K=int(rng.integers(MIN_K,MAX_K+1) if k is None else k)
    d=rng.dirichlet(np.full(K,1.8));t=np.r_[0,np.cumsum(d)];t[-1]=1
    a=np.zeros(K+1);scale=float(np.exp(rng.normal(np.log(25),.95)))
    a[2:K-1]=rng.normal(size=K-3)*scale
    return encode(t,a)

def initial_design(seed,n=64):
    rng=np.random.default_rng(seed);items=[]
    for K in [4,5,6,8,12,16]:
        if len(items)>=n:break
        t=np.linspace(0,1,K+1);a=np.zeros(K+1)
        a[2:K-1]=30*np.sin(np.linspace(0,np.pi,K-3))
        items.append(encode(t,a));items.append(encode(t,-a))
    items.append(encode(np.linspace(0,1,5),np.zeros(5)))
    while len(items)<n:items.append(sample_one(rng))
    return np.array(items[:n])

def mutate(x,rng,op=None):
    t,a=decode(x);K=len(t)-1
    if op is None:op=rng.choice(['split','delete','move_time','move_amp','rescale','reverse','crossover_like','restart'],p=[.17,.15,.14,.23,.11,.06,.08,.06])
    if op=='restart':return sample_one(rng),op
    if op=='split' and K<MAX_K:
        idx=int(rng.integers(1,K-1));new_t=t[idx]+rng.uniform(.25,.75)*(t[idx+1]-t[idx]);new_a=np.interp(new_t,t,a)
        t=np.insert(t,idx+1,new_t);a=np.insert(a,idx+1,new_a)
        a[min(K-1,max(2,idx+1))]+=rng.normal(0,max(1,np.max(np.abs(a))*.08))
    elif op=='delete' and K>MIN_K:
        idx=int(rng.integers(2,K-1));t=np.delete(t,idx);a=np.delete(a,idx)
    elif op=='move_time':
        idx=int(rng.integers(1,K));t[idx]=rng.uniform(t[idx-1]+.05*(t[idx+1]-t[idx-1]),t[idx+1]-.05*(t[idx+1]-t[idx-1]))
    elif op=='move_amp':
        idx=int(rng.integers(2,K-1));a[idx]+=rng.normal(0,max(1.,np.max(np.abs(a))*.2))
    elif op=='rescale':a[2:K-1]*=np.exp(rng.normal(0,.45))
    elif op=='reverse':a[2:K-1]*=-1
    elif op=='crossover_like':
        idx=int(rng.integers(2,K-1));a[2:idx+1]*=np.exp(rng.normal(0,.6))
    a[0]=a[1]=a[-2]=a[-1]=0
    return encode(t,a),op


In [ ]:
%%writefile APLC_experiment_runtime/oracle.py
"""Accelerated deterministic FerroSim GLD evaluator.

Physics: native rama_dev_updated@5eb258e, gamma=1, tetragonal, k=100,
dep_alpha=.3, finite-grid interactions with fixed /4 neighbor normalization.
Default integration is subdivided RK4, NOT the native first-order Euler method.
No field-amplitude bound or state clipping is imposed in RK4. The native
calDeriv clipping guard |P|=10 is explicitly diagnosed if ever crossed.
Native periodic central-difference curl and an interior diagnostic are returned.
"""
from __future__ import annotations
import os
import math
from pathlib import Path
import numpy as np
os.environ.setdefault("NUMBA_NUM_THREADS", "2")
from numba import njit, prange

N = 20
ALPHA1, ALPHA2, ALPHA3 = -1.6, 12.2, 40.0
COUPLING, DEP_ALPHA = 100.0, 0.3
EC = (-2.0 / 3.0) * ALPHA1 * np.sqrt(abs(-ALPHA1/(3.0*ALPHA2)))
TIME = np.linspace(0.0, 4.0, 1200)[:950]
DT = float(TIME[1]-TIME[0])
SOURCE_COMMIT = "5eb258e2a47d10b3d9bdb9e94dd9879b8410024c"
_LOCAL_DEFECT_PATH = Path(os.environ.get("FERROSIM_DEFECT_PATH", str(Path(__file__).with_name("defect_list_seed.npy"))))
DEFAULT_DEFECT_PATH = (_LOCAL_DEFECT_PATH if _LOCAL_DEFECT_PATH.exists()
                       else Path(__file__).resolve().parents[1]/"GA-DKL/defect_list_seed.npy")
_DEFECTS = None


def default_defects():
    global _DEFECTS
    if _DEFECTS is None:
        _DEFECTS = np.load(DEFAULT_DEFECT_PATH, allow_pickle=False).astype(np.float64).reshape(N,N,2)
    return _DEFECTS.copy()


@njit(cache=True)
def _rhs(p, ex, defects, coupling, dep_alpha, native_clip):
    n = p.shape[1]
    out = np.empty_like(p)
    mx, my = 0.0, 0.0
    for i in range(n):
        for j in range(n):
            mx += p[0,i,j]
            my += p[1,i,j]
    mx /= n*n
    my /= n*n
    for i in range(n):
        for j in range(n):
            x, y = p[0,i,j], p[1,i,j]
            if native_clip:
                x = min(10.,max(-10.,x))
                y = min(10.,max(-10.,y))
            sx, sy = 0.0, 0.0
            # Same finite-grid graph as native knn_search(r=1.1).
            if i>0:
                sx += p[0,i-1,j]; sy += p[1,i-1,j]
            if j>0:
                sx += p[0,i,j-1]; sy += p[1,i,j-1]
            if j<n-1:
                sx += p[0,i,j+1]; sy += p[1,i,j+1]
            if i<n-1:
                sx += p[0,i+1,j]; sy += p[1,i+1,j]
            out[0,i,j] = -(2*ALPHA1*x + 4*ALPHA2*x*x*x + 2*ALPHA3*y*y*x
                           + coupling*(x-sx/4.) - ex + dep_alpha*mx - defects[i,j,0])
            out[1,i,j] = -(2*ALPHA1*y + 4*ALPHA2*y*y*y + 2*ALPHA3*x*x*y
                           + coupling*(y-sy/4.) + dep_alpha*my - defects[i,j,1])
    return out


@njit(cache=True)
def _maxabs(p):
    q = 0.0
    for k in range(p.size):
        v = abs(p.flat[k])
        if not np.isfinite(v):
            return np.inf
        if v>q:
            q=v
    return q


@njit(cache=True)
def _curl(p):
    n=p.shape[1]
    c=np.empty((n,n),dtype=np.float64)
    total, interior = 0.,0.
    for i in range(n):
        for j in range(n):
            v=.5*(p[1,(i+1)%n,j]-p[1,(i-1)%n,j]-p[0,i,(j+1)%n]+p[0,i,(j-1)%n])
            c[i,j]=v
            total+=abs(v)
            if 0<i<n-1 and 0<j<n-1:
                interior+=abs(v)
    return total,interior,c


@njit(cache=True)
def _integrate(field, defects, initial, fidelity, method, history, coupling, dep_alpha, max_substeps):
    """method 0: native-style Euler, method 1: curvature-subdivided RK4."""
    p=initial.copy()
    steps=field.size
    n=p.shape[1]
    hst=np.full((2,steps,n,n),np.nan,dtype=np.float64) if history else np.empty((2,1,n,n),dtype=np.float64)
    hst[:,0,:,:]=p
    peak=_maxabs(p)
    defects_peak=0.
    for q in range(defects.size):
        defects_peak=max(defects_peak,abs(defects.flat[q]))
    rhs_calls=0
    subdivisions=0
    guard_exceeded=peak>10.
    status=0
    for t in range(1,steps):
        e0,e1=field[t-1]*EC,field[t]*EC
        if method==0:
            count=fidelity
        else:
            # A conservative estimate of the local Jacobian row-sum magnitude.
            # The forcing equilibrium scale anticipates sharp field increases.
            pbound=max(_maxabs(p),((max(abs(e0),abs(e1))+defects_peak+10.)/(4*ALPHA2))**(1/3)+.3)
            curvature=(12*ALPHA2+6*abs(ALPHA3))*pbound*pbound+2*coupling+abs(2*ALPHA1)+dep_alpha
            needed=DT*curvature*fidelity
            # Check in floating point before integer conversion: finite but huge
            # requested fields can otherwise overflow the subdivision counter.
            if not np.isfinite(needed) or needed>max_substeps:
                status=2
                break
            count=max(2*fidelity,int(math.ceil(needed)))
        if count>max_substeps:
            status=2
            break
        h=DT/count
        for sub in range(count):
            f0=sub/count
            f1=(sub+1)/count
            if method==0:
                # At fidelity=1 this exactly uses field[t], as native calcODE does.
                ex=e0+(e1-e0)*f1
                p=p+h*_rhs(p,ex,defects,coupling,dep_alpha,True)
                rhs_calls+=1
            else:
                ea=e0+(e1-e0)*f0
                eb=e0+(e1-e0)*f1
                em=.5*(ea+eb)
                k1=_rhs(p,ea,defects,coupling,dep_alpha,False)
                k2=_rhs(p+.5*h*k1,em,defects,coupling,dep_alpha,False)
                k3=_rhs(p+.5*h*k2,em,defects,coupling,dep_alpha,False)
                k4=_rhs(p+h*k3,eb,defects,coupling,dep_alpha,False)
                p=p+(h/6.)*(k1+2*k2+2*k3+k4)
                rhs_calls+=4
            pm=_maxabs(p)
            if not np.isfinite(pm):
                status=1
                break
            peak=max(peak,pm)
            guard_exceeded=guard_exceeded or pm>10.
        subdivisions+=count
        if history:
            hst[:,t,:,:]=p
        if status:
            break
    if not history:
        hst[:,0,:,:]=p
    if status:
        return np.nan,np.nan,p,peak,guard_exceeded,status,rhs_calls,subdivisions,hst
    reward,interior,_=_curl(p)
    return reward,interior,p,peak,guard_exceeded,status,rhs_calls,subdivisions,hst


@njit(cache=True,parallel=True)
def _batch(fields,defects,initial,fidelity,method,coupling,dep_alpha,max_substeps):
    batch=fields.shape[0]
    n=initial.shape[1]
    reward=np.empty(batch); interior=np.empty(batch); peak=np.empty(batch)
    final=np.empty((batch,2,n,n)); guards=np.empty(batch,dtype=np.bool_)
    status=np.empty(batch,dtype=np.int64); rhs=np.empty(batch,dtype=np.int64); subdivisions=np.empty(batch,dtype=np.int64)
    for q in prange(batch):
        r,j,p,m,g,s,nrhs,nsub,_=_integrate(fields[q],defects,initial,fidelity,method,False,coupling,dep_alpha,max_substeps)
        reward[q]=r; interior[q]=j; final[q]=p; peak[q]=m
        guards[q]=g; status[q]=s; rhs[q]=nrhs; subdivisions[q]=nsub
    return reward,interior,final,peak,guards,status,rhs,subdivisions


def evaluate_fields(fields_ec, fidelity=1, return_history=False, method="rk4", defects_ec=None,
                    initial_p=None, coupling=COUPLING, dep_alpha=DEP_ALPHA, max_substeps=100000):
    """Evaluate x-only waveforms with a fresh state for every candidate.

    Parameters
    ----------
    fields_ec : (batch,900) or (batch,950), or a single corresponding vector.
        Actual native-interface field E/Ec, WITHOUT normalization or amplitude cap.
        With 900 samples, 50 repeats of the final field are appended.
    fidelity : integer >=1
        Multiplies RK4 temporal resolution. Legacy Euler uses this many substeps.
    return_history : bool
        Add P_history shape (batch,2,950,n,n), mean_P and curl_history.
    method : "rk4" (default) or "legacy_euler"
        RK4 uses linear field interpolation and no state clipping.
    initial_p : optional (n,n,2) or (2,n,n), default zeros on20x20.
    defects_ec : optional (n*n,2) or (n,n,2), default published19-site array.

    status=0 success; 1 nonfinite integration; 2 subdivision resource limit.
    native_guard_exceeded flags |P|>10 without clipping or discarding the result.
    """
    arr=np.asarray(fields_ec,dtype=np.float64)
    if arr.ndim==1:
        arr=arr[None,:]
    if arr.ndim!=2 or arr.shape[1] not in (900,950):
        raise ValueError("Expected (batch,900) or (batch,950) native E/Ec values")
    if not np.isfinite(arr).all():
        raise ValueError("All supplied fields must be finite")
    if arr.shape[1]==900:
        arr=np.concatenate((arr,np.repeat(arr[:,-1:],50,axis=1)),axis=1)
    arr=np.ascontiguousarray(arr)
    if int(fidelity)!=fidelity or fidelity<1:
        raise ValueError("fidelity must be an integer >=1")
    if int(max_substeps)!=max_substeps or max_substeps<1:
        raise ValueError("max_substeps must be an integer >=1")
    if method not in ("rk4","legacy_euler"):
        raise ValueError("Unknown integration method")
    method_id=int(method=="rk4")
    if initial_p is None:
        initial=np.zeros((2,N,N),dtype=np.float64)
    else:
        initial=np.asarray(initial_p,dtype=np.float64)
        if initial.shape[-1]==2 and initial.shape[0]!=2:
            initial=initial.transpose(2,0,1)
        if initial.ndim!=3 or initial.shape[0]!=2 or initial.shape[1]!=initial.shape[2]:
            raise ValueError("Invalid initial polarization shape")
        initial=np.ascontiguousarray(initial)
    n=initial.shape[1]
    defects=default_defects() if defects_ec is None else np.asarray(defects_ec,dtype=np.float64).reshape(n,n,2)
    if defects.shape!=(n,n,2):
        raise ValueError("Defect grid must match initial polarization grid")
    if not np.isfinite(initial).all() or not np.isfinite(defects).all():
        raise ValueError("Initial polarization and defects must be finite")
    defects=np.ascontiguousarray(defects*EC)
    if return_history:
        out=[_integrate(x,defects,initial,int(fidelity),method_id,True,float(coupling),float(dep_alpha),int(max_substeps)) for x in arr]
        r,j,p,m,g,s,nrhs,nsub,hist=map(np.asarray,zip(*out))
    else:
        r,j,p,m,g,s,nrhs,nsub=_batch(arr,defects,initial,int(fidelity),method_id,float(coupling),float(dep_alpha),int(max_substeps))
    result={"reward":r,"curl":r,"interior":j,"final_P":p,"max_abs_P":m,
            "native_guard_exceeded":g,"status":s,"valid":(s==0)&np.isfinite(r),
            "rhs_calls":nrhs,"substeps":nsub,"field_peak_ec":np.max(np.abs(arr),axis=1),
            "fidelity":int(fidelity),"method":method}
    if return_history:
        result["P_history"]=hist
        result["mean_P"]=hist.mean(axis=(3,4))
        ch=np.empty((arr.shape[0],arr.shape[1]))
        ih=np.empty_like(ch)
        for b in range(arr.shape[0]):
            for t in range(arr.shape[1]):
                ch[b,t],ih[b,t],_=_curl(hist[b,:,t])
        result["curl_history"]=ch
        result["interior_history"]=ih
        result["time"]=TIME.copy()
        result["field_ec"]=arr
    return result


def curl_field(p):
    """Native periodic centered curl diagnostic for P shaped (2,n,n)."""
    return _curl(np.asarray(p,dtype=np.float64))[2]


In [ ]:
%%writefile APLC_experiment_runtime/dkl.py
"""Nonlinear neural feature kernel + exact GP trained on fixed waveform samples."""
import numpy as np
import torch
from torch import nn
class DKL(nn.Module):
    def __init__(self,seed=0):
        super().__init__();torch.manual_seed(seed)
        self.net=nn.Sequential(nn.Linear(65,32),nn.Tanh(),nn.Linear(32,8)).double()
        self.logl=nn.Parameter(torch.tensor(0.,dtype=torch.double));self.logsig=nn.Parameter(torch.tensor(0.,dtype=torch.double))
        self.lognoise=nn.Parameter(torch.tensor(-2.,dtype=torch.double));self.optim=torch.optim.Adam(self.parameters(),lr=.015)
        self.cache=None
    def features(self,w):
        # 64 sampled waveform values plus segment count
        return self.net(torch.asinh(w))
    def kernel(self,a,b):
        dist=torch.cdist(a,b).square().clamp_min(0);return torch.exp(2*self.logsig)*torch.exp(-dist/(2*torch.exp(2*self.logl)))
    def fit(self,X,y,steps=35):
        from representation import fields,complexity
        w=fields(X,64);k=complexity(X)['segments'].reshape(-1,1)
        z=np.c_[w/30.,k/16.];self.ycenter=float(np.mean(y));self.yscale=max(float(np.std(y)),1e-3)
        self.X=torch.tensor(z,dtype=torch.double);self.y=torch.tensor((np.asarray(y)-self.ycenter)/self.yscale,dtype=torch.double)
        self.train()
        for _ in range(steps):
            self.optim.zero_grad();f=self.features(self.X);K=self.kernel(f,f);noise=torch.exp(2*self.lognoise)+1e-5
            L=torch.linalg.cholesky(K+noise*torch.eye(len(X),dtype=torch.double))
            alpha=torch.cholesky_solve(self.y[:,None],L)
            loss=(.5*(self.y[:,None]*alpha).sum()+torch.log(torch.diag(L)).sum())/len(X)
            loss.backward();self.optim.step()
            with torch.no_grad():self.logl.clamp_(-4,4);self.logsig.clamp_(-4,4);self.lognoise.clamp_(-6,1)
        self.eval()
        with torch.no_grad():
            self.f=self.features(self.X);K=self.kernel(self.f,self.f);self.L=torch.linalg.cholesky(K+(torch.exp(2*self.lognoise)+1e-5)*torch.eye(len(X)))
            self.alpha=torch.cholesky_solve(self.y[:,None],self.L)
        return self
    def predict(self,X):
        from representation import fields,complexity
        if not len(X):return np.array([]),np.array([])
        w=fields(X,64);k=complexity(X)['segments'].reshape(-1,1)
        with torch.no_grad():
            x=torch.tensor(np.c_[w/30.,k/16.],dtype=torch.double);f=self.features(x);c=self.kernel(f,self.f)
            mean=(c@self.alpha).squeeze();sol=torch.linalg.solve_triangular(self.L,c.T,upper=False)
            std=(torch.exp(2*self.logsig)-sol.square().sum(0)).clamp_min(1e-9).sqrt()
            return np.atleast_1d(mean.numpy()*self.yscale+self.ycenter),np.atleast_1d(std.numpy()*self.yscale)


In [ ]:
%%writefile APLC_experiment_runtime/experiment.py
"""Independent adaptive piecewise-linear FerroSim benchmark. Run with --pilot for quick numerical verification."""
import os, argparse,sys,json,time
os.environ.setdefault('NUMBA_NUM_THREADS','1');os.environ.setdefault('OMP_NUM_THREADS','1');os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
from pathlib import Path
import numpy as np
import pandas as pd
from representation import MIN_K,MAX_K,decode,encode,fields,initial_design,mutate,complexity,sample_one,canonical
from oracle import evaluate_fields,default_defects
from dkl import DKL
ORACLE_METHOD=os.getenv('APLC_ORACLE_METHOD','rk4')
METHODS=['GA-DKL','QD-DKL','Sequential-GFlowNet-inspired-DKL','Denoising-diffusion-inspired-DKL']

def disorder_maps(out,n=20):
    out.mkdir(parents=True,exist_ok=True);ref=default_defects();active=ref.reshape(-1,2);active=active[np.any(active!=0,axis=1)]
    assert len(active)==19
    maps={'reference':ref};np.save(out/'reference.npy',ref)
    for j in range(n-1):
        rg=np.random.default_rng(20261122+j);locations=rg.choice(400,19,replace=False);d=np.zeros((400,2));d[locations]=active[rg.permutation(19)]
        maps[f'd{j:02d}']=d.reshape(20,20,2);np.save(out/f'd{j:02d}.npy',maps[f'd{j:02d}'])
    return maps

def propose(method,observed,rewards,rng,n):
    ix=np.argsort(rewards)[::-1];pool=observed[ix[:max(2,min(32,len(ix)))]]
    if method.startswith('QD'):
        metrics=complexity(observed)['segments'];chosen=[]
        for k in np.unique(metrics):
            z=np.flatnonzero(metrics==k);chosen.append(observed[z[np.argmax(rewards[z])]])
        pool=np.array(chosen)
    proposals=[];operators=[]
    for _ in range(n):
        if rng.random()<.12:
            z=sample_one(rng);op='restart'
        else:
            p=pool[int(rng.integers(len(pool)))].copy()
            if method.startswith('Sequential'):
                # policy-inspired sequential knot edits; NOT a trained trajectory-balance GFlowNet
                for _ in range(int(rng.integers(1,4))):p,op=mutate(p,rng)
                z=p;op='sequential_'+op
            elif method.startswith('Denoising'):
                # structure-preserving stochastic denoising toward a high-reward parent;
                # NOT a learned reverse-time diffusion model.
                z=p
                for j in range(3):
                    z,op=mutate(z,rng,op='move_amp' if j<2 else rng.choice(['move_time','split','delete']))
                    t,a=decode(z);tb,ab=decode(p);a[2:-2]=.75*a[2:-2]+.25*np.interp(t[2:-2],tb,ab);z=encode(t,a)
                op='iterative_denoising'
            else:z,op=mutate(p,rng)
        proposals.append(z);operators.append(op)
    return np.asarray(proposals),operators

def evaluate(x,defects,method=None,fidelity=1):
    method=method or ORACLE_METHOD
    result=evaluate_fields(fields(x,900),fidelity=fidelity,method=method,defects_ec=defects,max_substeps=100000)
    rew=np.asarray(result['reward']);valid=np.asarray(result['valid']) & ~np.asarray(result['native_guard_exceeded'])
    rew=np.where(valid,rew,np.nan)
    return rew,valid,np.asarray(result['interior']),np.asarray(result['status']),np.asarray(result['rhs_calls'])

def one_run(mapname,defects,method,seed,out,initial_count=64,generations=32,batch=32,candidates=256,dkl_steps=35):
    path=out/mapname/method.replace('/','_')/f'seed_{seed}';path.mkdir(parents=True,exist_ok=True)
    history=path/'history.csv';coords_path=path/'coordinates.npz'
    if history.exists() and coords_path.exists():
        df=pd.read_csv(history)
        with np.load(coords_path) as q:x=q['x'];
        if len(df)==initial_count+generations*batch:return df,x
    rng=np.random.default_rng(10000*seed+sum(map(ord,method))+int(mapname[1:]) if mapname.startswith('d') else 10000*seed+sum(map(ord,method)))
    x=initial_design(seed,initial_count);rewards=[];rows=[]
    for generation in range(generations+1):
        if generation==0:chosen=x;ops=['initial']*len(x);pred=np.full(len(x),np.nan);unc=pred.copy()
        else:
            good=np.isfinite(rewards);xx=x[good];yy=np.asarray(rewards)[good]
            if len(yy)<2:pool=initial_design(seed+generation,candidates);means=np.zeros(len(pool));std=np.ones(len(pool))
            else:
                surrogate=DKL(seed+generation)
                surrogate.fit(xx,yy,steps=dkl_steps)
                pool,ops_pool=propose(method,xx,yy,rng,candidates)
                means,std=surrogate.predict(pool)
            rank=np.argsort(-(means+2*std));sel=rank[:batch]
            chosen=pool[sel];pred=means[sel];unc=std[sel];ops=[ops_pool[i] for i in sel] if len(yy)>=2 else ['initial']*len(sel)
            x=np.vstack([x,chosen])
        ry,valid,interior,status,rhs=evaluate(chosen,defects)
        if generation==0:rewards=list(ry)
        else:rewards.extend(list(ry))
        cm=complexity(chosen)
        for j,z in enumerate(chosen):
            rows.append(dict(map=mapname,method=method,seed=seed,evaluation=len(rows)+1,generation=generation,
                reward=ry[j],valid=bool(valid[j]),interior=interior[j],status=int(status[j]),rhs_calls=int(rhs[j]),
                segments=int(cm['segments'][j]),effective_segments=int(cm['effective_segments'][j]),
                turns=int(cm['turns'][j]),peak_ec=float(cm['peak_ec'][j]),rms_ec=float(cm['rms_ec'][j]),
                operation=ops[j],predicted_mean=float(pred[j]),predicted_std=float(unc[j])))
        df=pd.DataFrame(rows);df.to_csv(history,index=False);np.savez_compressed(coords_path,x=x)
    return df,x

def run(out, maps, methods=METHODS,seeds=(0,1,2),initial_count=64,generations=32,batch=32,candidates=256,dkl_steps=35):
    all=[];start=time.time()
    for mname,defects in maps.items():
        for seed in seeds:
            for method in methods:
                print(f'[{time.time()-start:.1f}s] {mname} seed={seed} {method}',flush=True)
                df,_=one_run(mname,defects,method,seed,out,initial_count,generations,batch,candidates,dkl_steps);all.append(df)
    whole=pd.concat(all,ignore_index=True);whole.to_csv(out/'all_evaluations.csv',index=False)
    print('Measured evaluations:',len(whole),'valid:',int(whole.valid.sum()),'time:',round(time.time()-start,1),'sec',flush=True)
    return whole

if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--out',default='APLC_results');p.add_argument('--pilot',action='store_true');p.add_argument('--generations',type=int,default=32);p.add_argument('--initial',type=int,default=64);p.add_argument('--batch',type=int,default=32);p.add_argument('--candidates',type=int,default=256);p.add_argument('--seeds',type=int,default=3);p.add_argument('--maps',type=int,default=20);p.add_argument('--dkl-steps',type=int,default=35)
    args=p.parse_args();out=Path(args.out);out.mkdir(parents=True,exist_ok=True);maps=disorder_maps(out/'disorder_maps',args.maps)
    if args.pilot:args.maps=1;args.seeds=1;args.initial=8;args.generations=1;args.batch=4;args.candidates=16;args.dkl_steps=8;maps={'reference':maps['reference']}
    run(out,maps,seeds=range(args.seeds),initial_count=args.initial,generations=args.generations,batch=args.batch,candidates=args.candidates,dkl_steps=args.dkl_steps)


In [ ]:
%%writefile APLC_experiment_runtime/analysis.py
"""Evaluation-led analysis of the FerroSim adaptive linear control experiment."""
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from representation import fields,decode,complexity
from oracle import evaluate_fields
from experiment import METHODS

def read_results(out):
    p=Path(out)/'all_evaluations.csv'
    if not p.exists():raise FileNotFoundError(f'{p}: first execute the experiment')
    return pd.read_csv(p)

def get_x(out,row):
    p=Path(out)/str(row['map'])/str(row['method'])/f"seed_{int(row['seed'])}"/'coordinates.npz'
    with np.load(p) as z:return z['x'][int(row['evaluation'])-1].copy()

def summarize(df):
    rows=[]
    for (mp,m,seed),g in df.groupby(['map','method','seed']):
        valid=g[g.valid & np.isfinite(g.reward)]
        if valid.empty:continue
        best=valid.loc[valid.reward.idxmax()]
        rows.append({'map':mp,'method':m,'seed':seed,'best_reward':best.reward,'segments':int(best.segments),'effective_segments':int(best.effective_segments),'turns':int(best.turns),'eval_to_best':int(best.evaluation),'total_evaluations':len(g),'valid_evaluations':len(valid)})
    return pd.DataFrame(rows)

def pareto(g,col='effective_segments'):
    g=g[g.valid & np.isfinite(g.reward)];keep=[];best=-np.inf
    for k in sorted(g[col].unique()):
        s=g[g[col]==k];row=s.loc[s.reward.idxmax()]
        if row.reward>best+1e-10:keep.append(row);best=row.reward
    return pd.DataFrame(keep)

def learning_curves(df):
    fig,ax=plt.subplots(figsize=(9,5))
    for method in METHODS:
        data=df[(df.method==method)&(df['map']=='reference')]
        if data.empty:continue
        curves=[]
        for seed,g in data.groupby('seed'):
            g=g.sort_values('evaluation');curves.append(np.maximum.accumulate(g.reward.fillna(-np.inf).values))
            ax.plot(g.evaluation,curves[-1],alpha=.22,lw=1)
        if len(set(map(len,curves)))==1:
            z=np.asarray(curves);x=np.arange(1,z.shape[1]+1)
            ax.plot(x,np.nanmean(z,axis=0),label=method,lw=2)
            if len(z)>1:ax.fill_between(x,np.nanmean(z,axis=0)-np.nanstd(z,axis=0),np.nanmean(z,axis=0)+np.nanstd(z,axis=0),alpha=.1)
    ax.set(xlabel='Physical evaluations / method',ylabel='Best observed curl',title='Convergence on reference defect map');ax.grid(alpha=.2);ax.legend(fontsize=7);fig.tight_layout();return fig

def complexity_fronts(df):
    fig,ax=plt.subplots(figsize=(8,5))
    for m in METHODS:
        g=df[(df.method==m)&(df['map']=='reference')];p=pareto(g)
        if len(p):ax.step(p.effective_segments,p.reward,where='post',marker='o',label=m)
    ax.set(xlabel='Effective linear segments',ylabel='Best terminal curl',title='Observed reward–complexity Pareto fronts');ax.legend(fontsize=7);ax.grid(alpha=.2);fig.tight_layout();return fig

def selected_controls(df,out):
    fig,axs=plt.subplots(2,2,figsize=(12,7),sharex=True)
    for ax,method in zip(axs.flat,METHODS):
        g=df[(df.method==method)&(df['map']=='reference')&df.valid];
        if g.empty:continue
        best=g.loc[g.reward.idxmax()];x=get_x(out,best);t,a=decode(x)
        ax.plot(t,a,'o-',label=f"K={int(best.segments)}, R={best.reward:.4f}")
        ax.set(title=method,ylabel='E/Ec',xlabel='Normalized time');ax.legend(fontsize=8);ax.grid(alpha=.2)
    fig.suptitle('Best measured piecewise-linear controls');fig.tight_layout();return fig

def evolution_images(df,out):
    fig,axs=plt.subplots(4,2,figsize=(13,12),layout='constrained')
    for i,m in enumerate(METHODS):
        g=df[(df.method==m)&(df['map']=='reference')&(df.seed==0)].sort_values('evaluation')
        if g.empty:continue
        current=-np.inf;chosen=[];knots=[]
        for _,r in g.iterrows():
            if r.valid and np.isfinite(r.reward) and r.reward>current:
                current=r.reward;best=get_x(out,r)
            elif not chosen:best=get_x(out,r)
            chosen.append(fields(best[None,:],256)[0]);knots.append(int(decode(best)[0].size-1))
        a=np.asarray(chosen)
        axs[i,0].imshow(a,origin='lower',aspect='auto',extent=[0,1,1,len(a)],interpolation='nearest')
        axs[i,0].set(title=f'{m}: incumbent waveform',xlabel='Normalized time',ylabel='Evaluation')
        axs[i,1].step(np.arange(1,len(a)+1),knots)
        axs[i,1].set(title='Number of segments in incumbent',xlabel='Evaluation',ylabel='K',ylim=(3,17))
    return fig

def disorder_performance(df):
    s=summarize(df);fig,axs=plt.subplots(1,2,figsize=(13,4))
    for m,z in s.groupby('method'):
        g=z.groupby('map').best_reward.agg(['mean','std']).reset_index()
        axs[0].plot(range(len(g)),g['mean'],marker='.',label=m)
        # Two levels: variability across seeds and across disorder
        axs[1].scatter(z.segments,z.best_reward,s=24,alpha=.6,label=m)
    axs[0].set(xlabel='Disorder index',ylabel='Mean best reward',title='Disorder dependence');axs[0].legend(fontsize=6)
    axs[1].set(xlabel='Segments in winner',ylabel='Best reward',title='Complexity versus performance');axs[1].legend(fontsize=6)
    for ax in axs:ax.grid(alpha=.2)
    fig.tight_layout();return fig

def variance_decomposition(df):
    s=summarize(df);rows=[]
    for m,g in s.groupby('method'):
        means=g.groupby('map').best_reward.mean()
        within=g.groupby('map').best_reward.var(ddof=1)
        rows.append(dict(method=m,across_disorder_variance=float(means.var(ddof=1)) if len(means)>1 else np.nan,
            mean_within_disorder_seed_variance=float(within.mean()) if within.notna().any() else np.nan))
    return pd.DataFrame(rows)

def target_attainment(df,fraction=.95):
    rows=[]
    for mapname,g in df.groupby('map'):
        valid=g[g.valid & np.isfinite(g.reward)]
        if valid.empty:continue
        init=valid[valid.generation==0].reward.min();champion=valid.reward.max()
        target=init+fraction*(champion-init)
        for method,h in g.groupby('method'):
            for seed,r in h.groupby('seed'):
                ok=r[r.valid & (r.reward>=target)]
                rows.append(dict(map=mapname,method=method,seed=seed,fraction=fraction,
                    threshold=target,evaluations_to_target=float(ok.evaluation.min()) if len(ok) else np.nan))
    return pd.DataFrame(rows)

def selected_table(df,retention=.99):
    rows=[]
    for mp,g in df.groupby('map'):
        good=g[g.valid&np.isfinite(g.reward)]
        if good.empty:continue
        best=good.loc[good.reward.idxmax()];baseline=good[good.generation==0].reward.min()
        cutoff=baseline+retention*(best.reward-baseline)
        q=good[good.reward>=cutoff].sort_values(['effective_segments','segments','reward'],ascending=[True,True,False])
        simplest=q.iloc[0]
        for label,r in [('champion',best),('parsimonious',simplest)]:
            rows.append(dict(map=mp,selection=label,method=r.method,seed=r.seed,evaluation=r.evaluation,reward=r.reward,
                segments=r.segments,effective_segments=r.effective_segments,turns=r.turns,field_peak=r.peak_ec))
    return pd.DataFrame(rows)

def cross_transfer(df,out,maps,selections=None,oracle_method='rk4',fidelity=1):
    selected=selected_table(df) if selections is None else selections
    rows=[]
    for _,r in selected.iterrows():
        x=get_x(out,r);w=fields(x,900)
        for target,d in maps.items():
            v=evaluate_fields(w,defects_ec=d,method=oracle_method,fidelity=fidelity)
            rows.append(dict(source=r['map'],target=target,selection=r.selection,reward=float(v['reward'][0]),
                valid=bool(v['valid'][0] and not v['native_guard_exceeded'][0]),method=r.method))
    return pd.DataFrame(rows)

def transfer_heatmap(df,sel='champion'):
    t=df[(df.selection==sel)&df.valid].pivot_table(index='source',columns='target',values='reward')
    fig,ax=plt.subplots(figsize=(max(6,len(t)*.45),max(5,len(t)*.38)));im=ax.imshow(t,aspect='auto');fig.colorbar(im,ax=ax,label='Transferred curl')
    ax.set(xticks=range(len(t.columns)),xticklabels=t.columns,yticks=range(len(t.index)),yticklabels=t.index,title=f'{sel} cross-disorder transfer')
    plt.setp(ax.get_xticklabels(),rotation=90);fig.tight_layout();return fig

def validation(df,out,maps,method='rk4',fidelity=2,max_controls=8):
    selections=selected_table(df)
    selections=selections.iloc[:max_controls]
    rows=[]
    for _,r in selections.iterrows():
        x=get_x(out,r);d=maps[r['map']]
        v=evaluate_fields(fields(x,900),defects_ec=d,method=method,fidelity=fidelity)
        rows.append(dict(map=r['map'],selection=r.selection,reference_reward=r.reward,
            refined_reward=float(v['reward'][0]),refined_interior=float(v['interior'][0]),
            valid=bool(v['valid'][0]),guard_exceeded=bool(v['native_guard_exceeded'][0]),
            rhs_calls=int(v['rhs_calls'][0]),refinement_method=method,fidelity=fidelity))
    return pd.DataFrame(rows)

def physical_maps(df,out,maps):
    r=df[df.valid].loc[df[df.valid].reward.idxmax()];x=get_x(out,r);v=evaluate_fields(fields(x,900),defects_ec=maps[r['map']],method='rk4')
    p=v['final_P'][0];curl=np.abs(.5*(np.roll(p[1],-1,axis=0)-np.roll(p[1],1,axis=0)-np.roll(p[0],-1,axis=1)+np.roll(p[0],1,axis=1)))
    fig,axs=plt.subplots(1,3,figsize=(11,3.8));
    for ax,field,title in zip(axs,[p[0],p[1],curl],['P_x','P_y','Absolute curl']):
        im=ax.imshow(field,origin='lower');fig.colorbar(im,ax=ax);ax.set_title(title)
    fig.tight_layout();return fig

def export_all(df,out):
    out=Path(out);res=out/'analysis';res.mkdir(exist_ok=True)
    summarize(df).to_csv(res/'method_run_summary.csv',index=False)
    selected_table(df).to_csv(res/'selected_controls.csv',index=False)
    variance_decomposition(df).to_csv(res/'variance_components.csv',index=False)
    target_attainment(df).to_csv(res/'target_attainment.csv',index=False)
    for name,fn in [('learning_curves',lambda:learning_curves(df)),('pareto_fronts',lambda:complexity_fronts(df)),
                    ('champion_waveforms',lambda:selected_controls(df,out)),('waveform_evolution',lambda:evolution_images(df,out)),
                    ('disorder_performance',lambda:disorder_performance(df))]:
        fig=fn();fig.savefig(res/(name+'.png'),dpi=150);plt.close(fig)
    return res


## 3. Original embedded reference disorder map and source audit

In [ ]:
REF_B85='c-oun7wQ`j$;eQ~P_3SlTAW;@Zl$1ZlV+i=qoAIaUsO_*m=~X4l#&V(cT3DEP6dh=XCxM+0{I#y1_n9`Mw&VbwF+bcE{0JsARsI7O?HNjtHWS%PcGN8!ix894#OpUE6&gKjJ-L0{6i|p`+h9lcB8k$XilRI$jDo~(<az&#D=fq!$%=rXIw_p*DwWLGy3~pKKB|n@%uK&eo@W_kJ0oxLcv9uMND&#51#gb(VCv40=%@VpVc*8TsMolj@b0hXZTd}(D~t$-+ia*3B8W-av06YqktT+b~F3hU00n))6XykMtK`gZQA5BY~uGoilo|;)}Yb!Its`FS69yYQe7TMv-I<ocLmp>uF>(8p#pejU*CT$cC`I83Qz$6MT1~F'
refdata=zlib.decompress(base64.b85decode(REF_B85.encode()))
(ROOT/'defect_list_seed.npy').write_bytes(refdata)
print('Reference SHA256:',hashlib.sha256(refdata).hexdigest())
from representation import fields,decode,encode,mutate,initial_design,complexity,MIN_K,MAX_K
from oracle import evaluate_fields,default_defects
from experiment import disorder_maps,METHODS,run
from analysis import (read_results,summarize,pareto,learning_curves,complexity_fronts,selected_controls,evolution_images,
                      disorder_performance,variance_decomposition,target_attainment,selected_table,cross_transfer,
                      transfer_heatmap,validation,physical_maps,export_all,get_x)
print('Physics 20x20 defects:', int(np.count_nonzero(np.linalg.norm(default_defects(),axis=2))))

## 4. Geometry, constraints, and structural operations

For `K∈[4,16]`, the genotype encodes `K`, `K` normalized positive durations (softmax through relative logits), and active internal knot amplitudes. The first and last segments are strictly flat, matching Fourier/Legendre zero-field, zero-slope endpoints. Deactivated/padded coefficients have no effect and are zero in canonical encodings. Structural mutations include split and delete operations as well as time and amplitude changes. *K* is variable; the actual chromosome is stored in a fixed-size serialized array for efficient machine learning.

In [ ]:
rng=np.random.default_rng(123);X=initial_design(1,12)
assert fields(X,900).shape==(12,900)
for row in X:
 t,a=decode(row);assert np.all(np.diff(t)>0) and np.all(a[[0,1,-2,-1]]==0)
 assert np.allclose(encode(t,a),row)
 for op in ('split','delete','move_time','move_amp','rescale','reverse'):
  child,_=mutate(row,rng,op);tt,aa=decode(child)
  assert MIN_K<=len(tt)-1<=MAX_K and np.all(np.diff(tt)>0) and np.all(aa[[0,1,-2,-1]]==0)
fig,ax=plt.subplots(figsize=(9,4))
for k in [4,6,10,16]:
 row=initial_design(k,24);z=next(v for v in row if len(decode(v)[0])-1==k) if any(len(decode(v)[0])-1==k for v in row) else row[-1]
 t,a=decode(z);ax.plot(t,a,'o-',label=f'{len(t)-1} segments')
ax.set(xlabel='Normalized time',ylabel='Applied field / Ec',title='Piecewise-linear geometry and selectable knot counts');ax.legend();ax.grid(alpha=.2);plt.show()

## 5. Generate 20 matched physical disorder maps

The exact same seed-based construction as the other standalone notebooks: original reference plus 19 independently relocated 19-site defect maps.

In [ ]:
DISORDER_MAPS=disorder_maps(ROOT/'disorder_maps',20)
fig,axs=plt.subplots(4,5,figsize=(13,8),layout='constrained')
for ax,(name,defects) in zip(axs.flat,DISORDER_MAPS.items()):
 ax.imshow(np.linalg.norm(defects,axis=2),origin='lower',interpolation='nearest');ax.set(title=name,xticks=[],yticks=[])
plt.show();print('Maps:',len(DISORDER_MAPS))

## 6. Optimize every method, seed, and disorder map

**Pilot:** 1 map × 1 seed × 4 proposal strategies × (8 initial + 4 acquired) = **48 valid attempts**, Euler integrator. **Full:** 20 maps × 3 seeds × 4 strategies × (64 initial + 32×32 acquired) = **261,120 physical evaluations**, subdivided RK4 integrator. Each strategy receives the same 64-point initial design per seed and disorder; the DKL posterior is retrained on measured data. The full run is computationally expensive.

The default seed and map loop is sequential and writes each run to a dedicated directory. It can be restarted without rerunning completed runs, with the caveat that a partial run may be restarted from its initial design. The full experiment does **not** implement the original trained GFlowNet and diffusion generative models.

In [ ]:
RESULTS=ROOT/('pilot_results' if PILOT else 'full_results')
if RUN_EXPERIMENT:
    args=[sys.executable,str(ROOT/'experiment.py'),'--out',str(RESULTS)]
    if PILOT: args+=['--pilot']
    env=os.environ.copy();env.update({'APLC_ORACLE_METHOD':'legacy_euler' if PILOT else 'rk4',
        'OPENBLAS_NUM_THREADS':'1','NUMBA_NUM_THREADS':'1','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1'})
    print('Launching independent FerroSim experiment subprocess:', 'EULER PILOT' if PILOT else 'RK4 FULL')
    subprocess.run(args,cwd=str(ROOT),env=env,check=True)
df=read_results(RESULTS)
print('Evaluations',len(df),'valid',int(df.valid.sum()))
display(summarize(df))

## 7. Reward versus physical evaluations: individual seeds, means, and uncertainty

In [ ]:
fig=learning_curves(df);plt.show()

## 8. Time-to-target: 95% and 99% of observed reward improvement

Targets are empirically defined from the evaluated ensemble. Pilot estimates are not comparable with full-study times-to-target.

In [ ]:
display(target_attainment(df,.95));display(target_attainment(df,.99))

## 9. Reward versus waveform complexity: Pareto front and parsimony selection

`segments` counts encoded segments. `effective_segments` merges nearly collinear slope runs; `turns` measures prominent field reversals. The parsimony selector retains at least 99% of observed reward *improvement from the minimum initial reference reward*, and selects the simplest effective waveform.

In [ ]:
fig=complexity_fronts(df);plt.show()
display(selected_table(df))

## 10. Optimized waveform plots and knot-position statistics

In [ ]:
fig=selected_controls(df,RESULTS);plt.show()
fig,axs=plt.subplots(1,2,figsize=(11,4))
for _,row in selected_table(df).query("selection == 'champion'").iterrows():
 x=get_x(RESULTS,row);t,a=decode(x);axs[0].scatter(t[1:-1],np.full(len(t)-2,row.reward),s=20,alpha=.5)
 axs[1].hist(np.diff(t),bins=np.linspace(0,1,16),alpha=.3,label=row.method)
axs[0].set(xlabel='Normalized knot position',ylabel='Reward',title='Knot timing versus functionality')
axs[1].set(xlabel='Segment duration / total drive',ylabel='Count',title='Duration distribution');axs[1].legend(fontsize=6);plt.show()

## 11. Two-dimensional `imshow`: evolution of waveforms and structural complexity

In [ ]:
fig=evolution_images(df,RESULTS);plt.show()

## 12. Disorder dependence: seed-level and disorder-level variation

The pilot has one seed and one map, so variance components are undefined until the full experiment runs.

In [ ]:
display(variance_decomposition(df))
fig=disorder_performance(df);plt.show()

## 13. Winners and mutation-operation provenance

In [ ]:
display(df.groupby(['method','operation'],dropna=False).agg(evaluations=('reward','count'),best_reward=('reward','max')).reset_index())
fig,ax=plt.subplots(figsize=(11,4))
for method,g in df.groupby('method'):
 x=g.groupby('generation').segments.mean();ax.plot(x.index,x.values,marker='o',label=method)
ax.set(xlabel='Generation',ylabel='Mean encoded segment count',title='Evolution of average segment count');ax.legend(fontsize=7);ax.grid(alpha=.2);plt.show()

## 14. Cross-disorder transfer matrix, without reoptimization

Optional expensive physical re-evaluation of selected controls on every target defect map. Uses RK4 regardless of pilot mode. Normalization by each target map’s independently found baseline and best control is reported when both are available.

In [ ]:
if RUN_TRANSFER:
 active={k:DISORDER_MAPS[k] for k in sorted(df['map'].unique())}
 tr=cross_transfer(df,RESULTS,active,oracle_method='rk4')
 targets=df[df.valid].groupby('map').reward.agg(['min','max'])
 tr['target_baseline']=tr.target.map(targets['min']);tr['target_best']=tr.target.map(targets['max'])
 denominator=(tr.target_best-tr.target_baseline).replace(0,np.nan)
 tr['normalized_gain']=(tr.reward-tr.target_baseline)/denominator
 display(tr.head());display(transfer_heatmap(tr));tr.to_csv(RESULTS/'cross_disorder_transfer.csv',index=False)
else:print('Cross-disorder transfer deferred: set RUN_TRANSFER=True after the full experiment.')

## 15. Independent numerical refinement (fidelity 2) and physical polarization maps

The full optimizer uses RK4 fidelity 1. This cell checks selected waveforms at fidelity 2. Euler pilot results must **not** be conflated with this refined RK4 comparison. A second optional plot reports final P_x, P_y, and absolute spatial curl.

In [ ]:
if RUN_REFINEMENT:
 display(validation(df,RESULTS,DISORDER_MAPS,method='rk4',fidelity=2,max_controls=8))
else:print('Independent refinement disabled; enable RUN_REFINEMENT=True to calculate.')
if RUN_SPATIAL_MAP:
 fig=physical_maps(df,RESULTS,DISORDER_MAPS);plt.show()
else:print('Spatial RK4 validation disabled; enable RUN_SPATIAL_MAP=True to calculate.')

## 16. Export CSV tables and high-resolution figures

In [ ]:
export_path=export_all(df,RESULTS)
print('Saved actual measured observations, summary tables, plots, and serialized controls to:', export_path.resolve())

## 17. Interpretation and reproducibility boundaries

- **Physical comparison:** compare three representations only when they use the same defect maps, numerical integration, objective, acquisition budget, and validity rules.
- **Representation-specific freedom:** APLC has 4–16 segments, with variable knot times and amplitudes; Fourier and Legendre have 32 free continuous coefficients. The counts alone do not ensure matched function-space expressiveness.
- **Complexity:** distinguish encoded segments, effective noncollinear segments, prominent reversals, and actual degrees of freedom.
- **Algorithm labels:** GA-DKL and QD-DKL use an actual trained DKL reward model; the two additional native-topology proposers are *inspired* by, not mathematically identical to, trained GFlowNets/diffusion models.
- **Pilot findings:** 48 explicit-Euler evaluations on one map are suitable for software validation, not manuscript-level claims about optimization advantage or disorder generalization.
- **Full findings:** no full-ensemble findings are supplied until this notebook completes its full run and the numerical checks pass.
